# Move slow work off the request path

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 31 §31.1–31.3 · type: walkthrough*

**The promise:** you will define a Celery task, enqueue it with `.delay()` so your API returns *immediately*, make it survive transient failures with retry + exponential backoff and `acks_late`, and schedule a periodic run with Beat — all running free and offline, no Redis required.

## 🧠 Why this matters

A web request should be fast. But agentic work is often *slow* — a research run takes minutes, a corpus needs reindexing, a nightly report must be generated. Do that work *inside* the request and the user stares at a spinner while your server ties up a worker for the whole job.

The fix is the oldest trick in serious backend design: move slow or scheduled work *out* of the request and into a **task queue**. A producer drops a message and moves on; a pool of **workers** drains the queue elsewhere. That decoupling buys three things at once — **responsiveness** (the producer doesn't wait), **resilience** (work survives in the queue if a worker is down), and **elasticity** (add workers to clear a backlog). **Celery** is the standard engine for this in Python, and it is the spine your capstone's `workers/` will be built on.

## Objectives & prereqs

**By the end you can:**
- Define a Celery task and enqueue it with `.delay()` so the API returns before the work runs.
- Make a task survive transient failures with `max_retries` + exponential backoff and `acks_late`.
- Explain *why* `acks_late` forces tasks to be **idempotent** (the at-least-once reality).
- Register a periodic schedule with **Beat** using `crontab(...)`.

**Prereqs:** Ch 4 (async), Ch 29 (at-least-once delivery, idempotency, retries/backoff). Read book §31.1–§31.3 alongside this.

**Runs free & offline.** Everything here uses Celery's `task_always_eager=True` — tasks execute **inline, in this process**, with no broker and no worker. The real `redis://` broker + worker via `docker compose` is the `MOCK=0` path, documented in the setup cell. ⚠️ Eager mode *hides* real broker and ack behaviour — we call out where, so you don't mistake the demo for production.

In [ ]:
# Setup — imports, env, and the MOCK switch.
import os
import random
import time

from dotenv import load_dotenv

load_dotenv()

# MOCK=1 (default) runs Celery in EAGER mode: tasks execute inline, no Redis, no worker.
# MOCK=0 expects a real broker (e.g. redis://localhost:6379/0) and a running worker
#        started via `docker compose up` — see the template's compose (api + worker + Redis).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"
BROKER_URL = os.getenv("CELERY_BROKER_URL", "redis://localhost:6379/0")
RESULT_BACKEND = os.getenv("CELERY_RESULT_BACKEND", "redis://localhost:6379/1")

# Seed so the stubbed 'slow work' and its flaky-failure schedule are reproducible.
random.seed(31)

from celery import Celery
from celery.schedules import crontab

app = Celery("platform", broker=BROKER_URL, backend=RESULT_BACKEND)

if MOCK:
    # The whole point: no services needed. Tasks run synchronously, in-process.
    app.conf.task_always_eager = True
    app.conf.task_eager_propagates = True  # let task exceptions surface here

print("MOCK mode:", MOCK, "— EAGER, no broker/worker needed" if MOCK else f"— LIVE broker {BROKER_URL}")

## The rule, then the map

The working rule for background work, straight from the chapter epigraph:

> **If a user has to wait for it, and it doesn't have to happen now, it shouldn't happen now.**

Before Celery specifics, place it on the map. Queues and streams divide by *purpose*: **queues** hand a discrete task to one worker; **streams** are a durable, replayable log many consumers read.

In [ ]:
# A pocket decision map — queue vs stream by purpose (book §31.2).
OPTIONS = {
    "SQS":      ("queue",  "Managed AWS queue — simple, reliable task distribution, minimal ops."),
    "RabbitMQ": ("queue",  "Rich routing; Celery's classic broker for task queues."),
    "Redis":    ("queue",  "Lightweight, fast broker — often 'good enough' to start."),
    "Kafka":    ("stream", "High-throughput, replayable append-only log; many consumers, replay."),
}
for name, (kind, use) in OPTIONS.items():
    print(f"{name:9} [{kind:6}] {use}")

print("\nRule of thumb: discrete tasks → a QUEUE; many consumers need the same events / replay → a STREAM.")

## 🔧 Build the book's `reindex_documents` task

Celery's productive surface is tiny: **define** a task, **`.delay()`** it, and a worker runs it elsewhere. Here is the chapter's task — a slow `reindex` (stubbed as a seeded counter, no real I/O) decorated with the three settings that carry correctness weight: `bind=True` (so the task can reference `self` for retries), `max_retries=3`, and `acks_late=True`.

In [ ]:
class TransientError(Exception):
    """A failure that is expected to clear on its own — a brief network blip, a throttled
    dependency. In a distributed system these are GUARANTEED, not exceptional (Ch 29)."""


# A stand-in for the real slow work. No network, no disk — just a tiny deterministic
# 'cost' so the demo runs instantly while still *representing* a minutes-long reindex.
REINDEX_CALLS = {"count": 0}


def run_reindex(corpus_id: str) -> int:
    REINDEX_CALLS["count"] += 1
    time.sleep(0)  # stands in for the slow work; kept at 0 so CI stays fast
    docs = 1000 + sum(ord(c) for c in corpus_id) % 500  # deterministic 'doc count'
    return docs


@app.task(bind=True, max_retries=3, acks_late=True)
def reindex_documents(self, corpus_id: str):
    try:
        docs = run_reindex(corpus_id)            # the slow work, off the request path
        return {"corpus_id": corpus_id, "reindexed": docs}
    except TransientError as exc:
        # exponential backoff: 1s, 2s, 4s — don't hammer a struggling dependency.
        raise self.retry(exc=exc, countdown=2 ** self.request.retries)


print("Task registered:", reindex_documents.name)

### `.delay()` — enqueue and return *now*

In your API handler you call `.delay(...)` and return immediately. In production the call hands the message to the broker and comes back in microseconds; the worker runs the task later, elsewhere. In `MOCK=1` (eager) it runs inline — so you see a real result object, but the "instant return" is simulated.

In [ ]:
t0 = time.perf_counter()
async_result = reindex_documents.delay("kb-42")   # <- in the API handler; returns an AsyncResult
enqueue_ms = (time.perf_counter() - t0) * 1000

print(f"handler returned an AsyncResult in ~{enqueue_ms:.2f} ms — the HTTP response can go out now")
print("task id   :", async_result.id)
print("result    :", async_result.get(timeout=5))  # .get() blocks for the value; you'd usually poll
print("run_reindex was actually invoked", REINDEX_CALLS["count"], "time(s)")

**What you just saw.** The handler got an `AsyncResult` back in well under a millisecond — the API never waited for the reindex. ⚠️ In **eager** mode `.get()` returns instantly because the task already ran inline; against a real broker, `.get()` *blocks* until a worker finishes, which is exactly why production code **polls** or **subscribes** (WebSocket/SSE, Ch 25) instead of calling `.get()` on the request path.

## 🔮 Predict: the backoff sequence

`self.retry(exc=exc, countdown=2 ** self.request.retries)` schedules the next attempt. On the first failure `self.request.retries` is `0`, then `1`, then `2`.

**Predict:** what is the *delay sequence* (in seconds) before attempts 2, 3, and 4? Decide, then run the cell that computes the same expression for each retry index.

In [ ]:
from celery.exceptions import Retry

delays = [2 ** retries for retries in range(3)]  # retries = 0, 1, 2 (max_retries=3)
print("countdown before each retry:", delays, "seconds  ->  1s, 2s, 4s")


# Same task shape, observed end-to-end against a transient-then-succeed dependency.
@app.task(bind=True, max_retries=3, acks_late=True)
def flaky_reindex(self, corpus_id: str, attempt: int):
    if attempt < 3:  # fail twice with a transient error, then succeed on the 3rd
        countdown = 2 ** self.request.retries
        print(f"  attempt {attempt}: TransientError -> would retry in {countdown}s")
        try:
            raise TransientError("dependency throttled")
        except TransientError as exc:
            raise self.retry(exc=exc, countdown=countdown)
    print(f"  attempt {attempt}: succeeded")
    return {"corpus_id": corpus_id, "attempts": attempt}


# ⚠️ Eager mode does NOT auto-re-run a retried task — `self.retry()` raises `Retry`, which
# propagates. A REAL worker is the thing that catches that signal, waits `countdown`, and
# re-delivers. We model that worker loop here so the demo is honest about *who* retries:
def worker_runs(task, corpus_id, max_attempts=4):
    for attempt in range(1, max_attempts + 1):
        try:
            return task.apply(args=(corpus_id, attempt)).get()  # eager: runs inline
        except Retry:
            continue  # the worker's job: honour the retry signal (we skip the real sleep)
    raise RuntimeError("max retries exceeded")


print("\nrunning flaky_reindex (fails twice, then succeeds):")
print("final:", worker_runs(flaky_reindex, "kb-42"))

**What you just saw.** The true backoff is `1s, 2s, 4s` — each retry waits twice as long, so a briefly-struggling dependency gets room to recover instead of being hammered. Notice *who* did the retrying: `self.retry()` only **raises a signal**; the **worker** is what catches it, waits `countdown`, and re-delivers (we modelled that worker loop because ⚠️ eager mode does *not* auto-re-run a retried task — a place the demo and production differ). Two settings did the heavy lifting: **`max_retries` + backoff** absorbs the transient failures that are *guaranteed* in a distributed system, and **`acks_late=True`** tells Celery to acknowledge the message only *after* the task completes — so if the worker crashes mid-task, the broker re-delivers it instead of losing the work.

## Beat — schedule a periodic run

**Beat** turns Celery into your scheduler: cron, but written in Python and wired to your task code. Register the chapter's `nightly-reindex` to fire at 03:00 every day.

In [ ]:
app.conf.beat_schedule = {
    "nightly-reindex": {
        "task": reindex_documents.name,            # 'platform.reindex_documents'
        "schedule": crontab(hour=3, minute=0),     # 03:00 every day
        "args": ("kb-42",),
    },
}

entry = app.conf.beat_schedule["nightly-reindex"]
print("registered Beat schedule 'nightly-reindex':")
print("  task    :", entry["task"])
print("  schedule:", entry["schedule"], "(03:00 daily)")
print("  args    :", entry["args"])
print("\nIn production a `celery beat` process enqueues this task on the schedule;")
print("a worker then runs it — no human, no request involved.")

## ⚠️ Pitfall: `acks_late` means a task can run **more than once**

`acks_late=True` is what makes your queue *reliable* — but it has a sharp edge. A worker can finish the task and crash *before* it acknowledges the message. The broker, never having heard the ack, re-delivers it, and a second worker runs it again. That is **at-least-once** delivery (Ch 29), and it is not a bug you can configure away — it is the price of not losing work.

The cell below simulates that exact double-delivery against a *non-idempotent* task to make the danger concrete.

In [ ]:
# A non-idempotent side effect: every run charges the card again.
CHARGES = []  # pretend this is a payments ledger


@app.task(acks_late=True)
def charge_customer(customer_id: str, cents: int):
    CHARGES.append((customer_id, cents))   # ⚠️ no guard — runs its side effect every time
    return {"charged": cents, "customer": customer_id}


# Simulate acks_late re-delivery: the SAME message handed to a worker twice
# (worker A finished, crashed before acking → broker re-delivers to worker B).
charge_customer.delay("cust-7", 4200)
charge_customer.delay("cust-7", 4200)   # the re-delivery

print("ledger after an acks_late re-delivery:", CHARGES)
print(f"customer cust-7 was charged {len(CHARGES)} times — the second is a DUPLICATE")
print("\nFix (notebook 31-02): make the task idempotent — an idempotency key + 'already done?' check.")

**What you just saw.** One logical request, **two** charges — because `acks_late` plus a crash equals at-least-once, and the task had no guard. This is the single most common way task queues corrupt data. The lesson isn't "avoid `acks_late`" (you want reliable delivery); it's **design every task to be safe to re-run**. Notebook `31-02` builds that idempotency guard.

## 🎯 Senior lens

The instinct, the moment work goes async, is to reach for microservices. Resist it. A **modular monolith plus a task queue** handles the vast majority of "do this slow thing later" needs: one thing to deploy, debug, and reason about, with no network between your own components. The queue gives you the responsiveness, resilience, and elasticity you actually wanted; microservices add distributed-systems complexity (Ch 29) you mostly don't.

Extract a service only when a concrete force demands it — a component that must scale on its own, or a team that needs an independent deploy cadence. Until then: thin API in front, a fleet of Celery workers behind a queue. That single shape, not a clever framework, is what makes background work production-grade.

## Recap

- **Move slow/scheduled work off the request path** into a task queue — you buy responsiveness, resilience, and elasticity.
- **Celery's surface is small:** `@app.task` to define, `.delay()` to enqueue (returns instantly), a worker runs it elsewhere.
- **`max_retries` + exponential backoff** (`countdown=2 ** retries` → 1s, 2s, 4s) absorbs the transient failures that are guaranteed in a distributed system.
- **`acks_late=True`** acks only after completion → no lost work on a crash, but **at-least-once**, so tasks can run **more than once**.
- **→ Therefore tasks must be idempotent** (next notebook). **Beat** schedules periodic runs via `crontab(...)`.
- **Queues vs streams:** discrete tasks → queue (SQS/RabbitMQ/Redis); replayable events for many consumers → stream (Kafka).
- **You don't need microservices** for background work — a modular monolith + a queue is the default.

## Exercises

Each *changes something* and asks you to predict the result first.

1. **Tune the backoff.** Change the retry to `countdown=min(2 ** self.request.retries, 8)` (capped backoff). Predict the new delay sequence for `max_retries=5`, then compute it. Why cap it at all?
2. **Watch a retry exhaust.** Make `flaky_reindex` fail on *every* attempt. Predict what Celery raises once `max_retries` is exceeded, then catch it and assert the task ran exactly `max_retries + 1` times.
3. **Add a second Beat job.** Register a `hourly-healthcheck` schedule with `crontab(minute=0)` (top of every hour) pointing at a tiny new task. Predict how many times it would fire in a day, then verify the `crontab` matches your intent.
4. **Eager vs broker.** Explain in a markdown cell one behaviour that `task_always_eager=True` *hides* relative to a real Redis broker + worker (hint: acks, ordering, or concurrency).

In [ ]:
# Exercise 1 — your code here.


In [ ]:
# Exercise 2 — your code here.


In [ ]:
# Exercise 3 — your code here.


## Next

- **Next notebook:** [`31-02-idempotent-tasks-and-orchestration-choices.ipynb`](31-02-idempotent-tasks-and-orchestration-choices.ipynb) — fix the `acks_late` double-run, then learn to *choose* between a queue, durable execution, and a DAG.
- **Book:** §31.1 (monolith vs microservices), §31.2 (queues & streaming), §31.3 (Celery in depth).
- **Blueprint this feeds:** [`blueprints/agent-loop/`](../../blueprints/agent-loop/) — whose runs become the retried, checkpointed Celery tasks of the next notebooks.
- **Template:** [`templates/fastapi-agent-service/`](../../templates/fastapi-agent-service/) — the `docker compose` (api + worker + Redis) that turns `MOCK=0` into a real broker.
- **Capstone:** this is where `capstone-project/workers/` is born — the Celery app, tasks, and Beat schedules you just sketched.